# Zig Zag Indicator：用 qust 标注显著转折

来源参考：[Investopedia](https://www.investopedia.com/terms/z/zig_zag_indicator.asp)


这篇 notebook 按 Investopedia 原页面的信息结构做完整中文改写，并把指标定义落成 `col(...).investopedia.xxx(...)` 的一行调用。能用 qust 现有 rolling、shift、select、with_cols、over 组合的就直接组合；需要 pivot/形态扫描的部分由 Rust helper 完成，Python 端不写 UDF。


## 1. Investopedia 原文内容完整改写：Zig Zag Indicator

### 什么是 Zig Zag Indicator
Zig Zag 指标是一种价格过滤和结构标注工具。它不会试图预测下一根 K 线，而是把小于某个阈值的价格波动忽略掉，只保留足够大的转折。图上看到的 Zig Zag 线，其实是在连接一系列被确认的显著高点和显著低点。阈值通常用百分比表示，例如 5% 表示价格必须从前一个极值反向走出至少 5%，那个极值才被确认为一个转折点。

### 它为什么有用
普通价格图会包含大量噪声，尤其在低周期数据里，连续的小波动会遮住真正重要的趋势段。Zig Zag 的作用是把这些细小波动压下去，让交易者更容易观察大级别的波段、高低点结构、趋势线、支撑阻力以及可能的反转区域。它常被用作复盘和辅助画线工具，而不是独立交易信号。

### 计算逻辑
计算时先选择一个价格输入，可以是 close，也可以用 high/low/close 让高低点更贴近 K 线真实区间。算法会维护当前方向和当前方向上的极值：上涨段里持续刷新最高点，直到价格从最高点回落超过阈值；下跌段里持续刷新最低点，直到价格从最低点反弹超过阈值。达到反向阈值时，前一个极值被确认并画到 Zig Zag 线上。

### 如何解读
Zig Zag 线向上连接低点到高点，说明这一段被视为主要上涨波段；向下连接高点到低点，说明这一段被视为主要下跌波段。交易者常把这些节点拿来辅助识别更高的高点、更高的低点、更低的高点、更低的低点，也会用它观察价格是否形成双顶、双底、通道或楔形。

### 参数影响
阈值越小，Zig Zag 越敏感，转折点越多，也更容易受噪声影响；阈值越大，转折点越少，线条更干净，但确认更慢。这个参数没有固定正确值，要和品种波动率、周期、研究目的匹配。

### 局限性
Zig Zag 最大的问题是最后一段会变化。因为转折需要未来价格确认，所以最近一个潜在 pivot 在确认前可能消失或移动。它适合做结构分析和研究标注，但直接用于实时交易时必须把确认延迟计算进去，不能把尚未确认的节点当作当时已经知道的信息。

## 2. 从文章到 qust 算子的落地

Investopedia 的核心是“百分比过滤 + 确认转折”。qust 的 `investopedia.zig_zag` 按这个语义输出三列：确认 pivot 的价格、pivot 类型、当前趋势方向。输出和输入同长度，便于和 K 线一起画，也便于后续组合其他形态识别。

## 3. qust 一行调用

```python
col("high", "low", "close").investopedia.zig_zag(percent=3.0)
```

输入列顺序：`high, low, close`；也可以只传 `close`。

输出列：`zig_zag`, `zig_zag_pivot`, `zig_zag_trend`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import os
import sys

LOCAL_QUST_SOURCE = "/root/otters/otters-py/python"
if os.path.isdir(LOCAL_QUST_SOURCE) and LOCAL_QUST_SOURCE not in sys.path:
    sys.path.insert(0, LOCAL_QUST_SOURCE)

import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "/root/qust-py/examples/data/data_kline3.parquet"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实本地 K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("high", "low", "close").investopedia.zig_zag(percent=3.0)
zigzag_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
zigzag_data = zigzag_data.with_columns((pl.col("zig_zag_pivot") != 0).alias("zig_zag_signal"))
plot_data = (
    zigzag_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    (col("zig_zag_pivot") == col.lit(1)).cast(pl.UInt32).sum().alias("pivot_high_count"),
    (col("zig_zag_pivot") == col.lit(-1)).cast(pl.UInt32).sum().alias("pivot_low_count"),
).calc_data(zigzag_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 12)


pivot_high_count,pivot_low_count
u32,u32
525,518


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 notebook 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
zigzag_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("zigzag_price", show_axis_label=True)
        .kline(),
    col("datetime", "zig_zag")
        .monitor("zigzag_price", show_axis_label=True)
        .line(),
    col("datetime", "zig_zag", "zig_zag_signal")
        .monitor("zigzag_price", show_axis_label=True)
        .mark(shape=mark_shape.circle, color="#ff6b6b", width=0.35),
).monitor.make_monitor("black").monitor.add_grid([
    ["zigzag_price"],
]).runtime()

zigzag_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Zig Zag 策略回测

`zig_zag_pivot` 是确认后回写到历史极值行的展示列，直接交易会有未来函数。策略不能用它入场。这里改用当前行可见的 `zig_zag_trend` 变化：趋势从 -1 切到 1 作为多头确认，从 1 切到 -1 作为空头确认；信号再后移一根 K 线交易。

In [5]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015


def make_two_sided_strategy(indicator_cols, open_long_raw, open_short_raw):
    """用当前指标生成完整多空策略；持仓用 fp.vol_pms 做品种/波动率尺度归一化。"""
    return (
        col
        .with_cols(indicator_cols)
        .with_cols(
            open_long_raw.fill_null(col.lit(False)).alias("open_long_raw"),
            open_short_raw.fill_null(col.lit(False)).alias("open_short_raw"),
        )
        # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
        .with_cols(
            col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
            col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
            col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
            col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
            col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
        )
        .with_cols(
            (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
                .fill_null(col.lit(False))
                .alias("exit_long_sig"),
            (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
                .fill_null(col.lit(False))
                .alias("exit_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
                .stra.to_hold_two_sides()
                .expanding()
                .alias("hold")
        )
        .with_cols(
            (col("hold") / col.all.fp.vol_pms()).alias("hold")
        )
        .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
        .over("ticker", "ct")
        .select(
            col("pnl")
                .sum()
                .group_by(col("datetime").dt.date().alias("date"))
                .batch.sort("date")
                .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
                .select("date", "pnl", "pnl_cum")
        )
    )


def calc_strategy_stats(strategy_daily: pl.DataFrame) -> pl.DataFrame:
    return col(
        col("date").first_value().alias("start_date"),
        col("date").last_value().alias("end_date"),
        col.lit(1).sum().alias("days"),
        col("pnl").sum().alias("total_pnl"),
        col("pnl").mean().alias("mean_daily_pnl"),
        col("pnl").std().alias("std_daily_pnl"),
        (col("pnl").mean() / col("pnl").std() * col.lit(252 ** 0.5)).alias("sharpe_like"),
        col("pnl").min().alias("worst_day_pnl"),
        col("pnl").max().alias("best_day_pnl"),
    ).calc_data(strategy_daily)

indicator_cols = col("high", "low", "close").investopedia.zig_zag(percent=3.0)
trend_turn_long = (col("zig_zag_trend") == col.lit(1)) & (col("zig_zag_trend").shift(1).expanding() == col.lit(-1))
trend_turn_short = (col("zig_zag_trend") == col.lit(-1)) & (col("zig_zag_trend").shift(1).expanding() == col.lit(1))
strategy_daily_expr = make_two_sided_strategy(
    indicator_cols,
    trend_turn_long,
    trend_turn_short,
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = calc_strategy_stats(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


start_date,end_date,days,total_pnl,mean_daily_pnl,std_daily_pnl,sharpe_like,worst_day_pnl,best_day_pnl
date,date,i32,f64,f64,f64,f64,f64,f64
2022-01-04,2024-12-31,859,8.628182,0.010115,0.789663,0.203343,-3.50661,6.101257


In [6]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,-0.437628,7.9169
2024-12-19,-0.848752,7.068148
2024-12-20,2.117104,9.185252
2024-12-21,0.0,9.185252
2024-12-23,-1.010351,8.174901
2024-12-24,-0.05432,8.120581
2024-12-25,0.108415,8.228996
2024-12-26,0.063541,8.292537
2024-12-27,-0.208081,8.084456


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。